In [ ]:
# Imports
import os
import sys
# ----File Stitching----
# If in results_cnn folder, cd back to MamalakisResearch folder
if os.path.basename(os.getcwd()) == "results_cnn":
    os.chdir('..')
# If a file is in /data_prep_viz/prep/, access it by telling the system to look at that path as well as current path
sys.path.append(os.path.join(os.getcwd(), '..', 'data_prep_viz/prep'))

In [ ]:
%run "data_prep_viz/prep/get_cnn_data_v3_sk.ipynb" # also manually typed in cells below

In [20]:
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
import netCDF4 as nc

In [16]:
# pulling variables out for the plot function 
VAR_LIST = ["tas", "tasmax", "tasmin", "pr", "psl", "sfcWind", "mrsos"]

# dictionary for each unit for the plot 
UNIT_MAP = {
    "tas": "°C", 
    "tasmax": "°C", 
    "tasmin": "°C", 
    "pr": "mm/day", 
    "psl": "hPa", 
    "sfcWind": "m/s", 
    "mrsos": "kg/m²"}

In [17]:
def convert_units(varname: str, x: np.ndarray):
    """
    varname: index number from the list of variables so get_data func can convert units 
    x: data that needs units changed (raw x data) in get_data func 
    """
    if varname in {"tas", "tasmax", "tasmin"}:
        # kelvin to celcius
        return x - 273.15, "$^{\circ}$C"
    if varname == "pr":
        # kg/m2/s to mm/day
        return x * 86400.0, "mm/day"
    if varname == "psl":
        # pascals to hpa
        return x / 100.0, "hPa"
    
    # these don't need to be converted -- just adding the units 
    if varname == "sfcWind":
        return x, "m/s"
    if varname == "mrsos":
        return x, "kg/m$^{2}$"
    return x, "unknown"




def get_model_name(path: str) -> str:
    """
    helper function so that get_cnn_tensors prints out the models that are being processed
    pulled from hayeon's code 
    """
    # Everything before "_ssp..."
    return os.path.basename(path).split("_ssp")[0]

In [18]:
def get_cnn_tensors(model_list, scenario, data_path, 
                    st_early=2015, end_early=2024, 
                    st_late=2050, end_late=2059,
                    stat='mean', use_anomaly=True, 
                    models_to_run=None,
                    vars_to_use=None):
    
    """
    model_list: variable of list of models
    scenario: input as either 'ssp119' or 'ssp126' strings
    data_path: variable of data_path for user
    st_early: early period start yr integer
    end_early: early period end yr integer
    st_late: late period start yr integer
    end_late: late period end yr integer
    stat: what stat function user wants to run to summarize variables 
        mean: mean
        std: standard deviation
        max: max val
        min: min val
        medium: median
    use_anomaly: whether to subtract values from baseline values to find anomaly
        True: subtract values
        False: don't subtract values 
    models_to_run: can specify number of models to run by index of model_list variable
        None: all models 
        [x]: one model to run 
        [x, x, x...]: whatever number of models to run 
    file_start_year: ensuring that if time dimension in models starts at 0 or 1, the func will slice the time correctly 
    vars_to_use: give list of strings of vars to calculate, if none specified (aka default) then does all 7 vars
    """
    # defining variable list for unit conversion later 
    var_list = ["tas", "tasmax", "tasmin", "pr", "psl", "sfcWind", "mrsos"]
    # default of calculating with all 7 vars
    if vars_to_use is None:
        selected_vars = var_list
    # if user specified certains vars to be calculated
    else:
        # check to see that the string items provided do exist
        selected_vars = [v for v in vars_to_use if v in var_list]

    # find the indexes of the specified variables defined by user (or defaulting to finding all the vars' indices)
    var_indices = [var_list.index(v) for v in selected_vars]

    # mapping out stats and the relevant func
    stat_map = {'mean': np.nanmean, 'std': np.nanstd, 'max': np.nanmax, 'min': np.nanmin, 'median': np.nanmedian}
    # variable that will calculate the stat for whatever the user wants and then will default to mean 
    calc_func = stat_map.get(stat.lower(), np.nanmean)

    # runs all models 
    if models_to_run is None:
        selected_models = model_list
    # runs one model if user only specified one
    elif isinstance(models_to_run, int):
        selected_models = [model_list[models_to_run]]
    # if none of the above options (all or one) was selected, then was list of models so get those models 
    else:
        selected_models = [model_list[i] for i in models_to_run]

    # initializing lists 
    x_list, y_list = [], []

    # opening file 
    for filename in selected_models:
        full_path = os.path.join(data_path, filename)
        model_short_name = get_model_name(filename)
        print(f"processing model: {model_short_name}")
        
        with nc.Dataset(full_path) as ds:
            # loading the data for specific model 
                # slicing dimensions for the ensembles, the specific var indices and all the times/long/lat dimensions
            data_all_vars = ds[f"data_{scenario}"][:, var_indices, :, :, :] 


            # starting unit conversions for ALL variables
            # looping through all 7 vars or the selected vars only 
                # using enumerate to keep track of the index of var and what the var is in the var_list defined above
                    # idx to keep track of what slice of the var dimension 
                    # var_name so that convert_units can be called correctly 
            for idx, var_name in enumerate(selected_vars):
                # converting var to relevant unit from func -- slicing the relevant var one at a time (getting all the info for all the other dimensions, just associated with that certain var)
                    # returns the converted numbers and string that gives converted unit
                converted_data, _ = convert_units(var_name, data_all_vars[:, idx, :, :, :])
                 # taking all the converted data and making it the 'all data' version for the var index 
                data_all_vars[:, idx, :, :, :] = converted_data
         
            # slice early period once per climate model file
            baseline_slice = data_all_vars[:, :, 0:120, :, :]
            
            
            # average ensembles (0 dimen) and months (2 dimen)
            p_mean = np.nanmean(baseline_slice, axis=(0, 2)) 
            p_std = np.nanstd(baseline_slice, axis=(0, 2))
            # if the standard deviation is less than super small, make it 1.0
            p_std[p_std < 1e-4] = 1.0

            # making list of periods where early period assigned 0 and late assigned 1 
            periods = [(st_early, end_early, 0), (st_late, end_late, 1)]

            # going thru periods 
            for start_yr, end_yr, label in periods:
                # making var for the number of ensembles are getting looked at 
                n_ens = data_all_vars.shape[0]
                
                # going thru every yr within each period 
                for yr_idx in range(start_yr, end_yr + 1):
                    # indexing by month 
                    m_idx_start = (yr_idx - 2015) * 12
                    m_idx_end = m_idx_start + 12
                    
                    # going thru individual ensemble for current yr 
                    for ens_idx in range(n_ens):
                        # slicing to get the info at the ensemble index, all vars, time index, all lats/longs
                        annual_slice = data_all_vars[ens_idx, :, m_idx_start:m_idx_end, :, :]
                        # applying user specified stat func to make the monthly vals into aggregated yearly vals
                        yearly_val = calc_func(annual_slice, axis=1) 
                        
                        if use_anomaly:
                            # calculate val from subtracting the value from the above calculated mean and then deivde by standard deviation
                            val = (yearly_val - p_mean) / p_std
                        else:
                            val = yearly_val
                        
                        x_list.append(val)
                        y_list.append(label)

    return np.nan_to_num(np.array(x_list), nan=0.0), np.array(y_list).reshape(-1, 1)

In [2]:
def get_gradients(inputs, model, top_pred_idx=None):
    """Computes the gradients of outputs w.r.t input image.

    Args:
        inputs: 2D/3D/4D matrix of samples
        top_pred_idx: (optional) Predicted label for the x_data
                      if classification problem. If regression,
                      do not include.

    Returns:
        Gradients of the predictions w.r.t img_input
    """
    inputs = tf.cast(inputs, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(inputs)
        
        # Run the forward pass of the layer and record operations
        # on GradientTape.
        preds = model(inputs, training=False)  
        
        # For classification, grab the top class
        if top_pred_idx is not None:
            preds = preds[:, top_pred_idx]
        
    # Use the gradient tape to automatically retrieve
    # the gradients of the trainable variables with respect to the loss.        
    grads = tape.gradient(preds, inputs)
    return grads

In [3]:
def get_integrated_gradients(inputs, model, baseline=None, num_steps=50, top_pred_idx=None):
    # 1. Ensure inputs and baseline are float32
    inputs = inputs.astype(np.float32)
    
    if baseline is None:
        # Fallback to zeros if no baseline provided
        baseline = np.zeros_like(inputs).astype(np.float32)
    else:
        baseline = baseline.astype(np.float32)
        # Ensure baseline has a leading dimension if it's a single mean map
        if baseline.ndim == inputs.ndim - 1:
            baseline = np.expand_dims(baseline, axis=0)

    # 2. Generate interpolation steps
    # We use np.linspace to create the scaling factors (alphas)
    alphas = np.linspace(0.0, 1.0, num_steps + 1)
    
    # 3. Compute Gradients along the path
    # We iterate through the interpolation path from baseline to input
    all_grads = []
    for alpha in alphas:
        # Interpolate: baseline + alpha * (input - baseline)
        step_input = baseline + alpha * (inputs - baseline)
        
        # Get gradients for this specific step
        grad = get_gradients(step_input, model, top_pred_idx=top_pred_idx)
        all_grads.append(grad)
    
    # 4. Convert to tensor for averaging
    # Shape: (num_steps + 1, batch, vars, lat, lon)
    all_grads = tf.convert_to_tensor(all_grads, dtype=tf.float32)

    # 5. Approximate the integral (Trapezoidal Rule)
    # Average the gradients of adjacent steps
    grads_at_step_ends = (all_grads[:-1] + all_grads[1:]) / 2.0
    avg_grads = tf.reduce_mean(grads_at_step_ends, axis=0)

    # 6. Final IG calculation: (input - baseline) * average gradient
    integrated_grads = (inputs - baseline) * avg_grads.numpy()
    
    return integrated_grads

In [ ]:
def cnn_training(X_data, y_data, learning_rate=0.001, epochs=200, batch_size=64):
    # prep indices
    n_samples = X_data.shape[0]
    indices = np.arange(n_samples) # [0, 1, 2, ..., N-1]

    X = np.transpose(X_data, (0, 2, 3, 1))  # (N, lat, lon, 7)
    y = y_data.astype(np.float32)           # (N, 1)

    
    # split test set (50 samples) 
        # passing indices to keep track of the indices that are goin in the set 
    X_rem, X_test, y_rem, y_test, idx_rem, test_indices = train_test_split(
        X, y, indices,
        test_size=50,
        stratify=y
    )

    # val split (from remaning 450 samples)
    X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
        X_rem, y_rem, idx_rem,
        test_size=50,
        stratify=y_rem
    )

    
    lat, lon = X_train.shape[1], X_train.shape[2]
    model = models.Sequential([
        layers.Input(shape=(lat, lon, 7)),
        
        #  CNN block (64 filters) with two convs, then pool
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        # CNN block 32 kernels (conv + pool)
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        # CNN block 16 kernels (conv only)
        layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(16, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(8, (3, 3), padding="same", activation="relu"),
        layers.Conv2D(8, (3, 3), padding="same", activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),             
        layers.Dense(50, activation="relu"),
        layers.Dense(10, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=20,
        restore_best_weights=True,
        verbose=0
    )

    # train model 
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )
    
    # Return everything needed for the large loop
    return model, X_train, y_train, X_test, y_test, test_indices

In [ ]:

import xarray as xr

def run_climate_experiment(scenarios, early_starts, model_list, data_path):

    # initializing list to store results from each early/late period iteration
    all_results = []
    
    # looping thru each scenario (ssp119, ssp126)
    for scenario in scenarios:
        # for every early start year in early_starts list
        for early_start in early_starts:
            # make late period start years as 10 plus the early start year, going up to 2095
            late_starts = np.arange(early_start + 10, 2095, 10) 
            
            for late_start in late_starts:
                print(f"Processing: {scenario} | Early: {early_start} | Late: {late_start}")
                
                # prepping data for every early and late 10yr time period combo 
                X_data, y_data = get_cnn_tensors(
                    model_list, scenario, data_path, 
                    st_early=early_start, end_early=early_start+9, 
                    st_late=late_start, end_late=late_start+9
                )
                
                # training data 
                model, X_train, y_train, X_test, y_test, test_idx = cnn_training(X_data, y_data)
                
                # predicting
                preds = model.predict(X_test).flatten()
                
                # XAI STUFF: 
                # baseline is the mean of early period from training set
                early_idx = np.where(y_train == 0)[0]
                baseline = np.mean(X_train[early_idx], axis=0, keepdims=True)
                
                # getting late indices for X_test set 
                late_test_idx = np.where(y_test == 1)[0]
                ig_samples = X_test[late_test_idx]
                
                # integrated gradient calculation based on the early period baseline on the late period stuff 
                ig_output = get_integrated_gradients(ig_samples, model, baseline)
                if hasattr(ig_output, 'numpy'): 
                    ig_output = ig_output.numpy()

             
                # storing all the iteration data as a dict 
                iteration_data = {
                    'scenario': scenario,
                    'early_yr': early_start,
                    'late_yr': late_start,
                    'y_true': y_test,
                    'y_pred': preds,
                    'test_indices': test_idx,
                    'ig_heatmaps': ig_output 
                }
                # appending everything to the all_results list 
                all_results.append(iteration_data)

    # --- SAVING DATA ---
    save_results(all_results)
    return all_results

def save_results(results_list, filename="experiment_results.nc"):
    """
    Saves the nested results into a NetCDF file, which is ideal for 
    high-dimensional climate heatmaps.
    """
    # For a quick CSV of just the performance:
    summary_df = pd.DataFrame([{
        'scenario': r['scenario'],
        'early': r['early_yr'],
        'late': r['late_yr'],
        'mean_pred': np.mean(r['y_pred'])
    } for r in results_list])
    summary_df.to_csv("experiment_summary.csv", index=False)
    
    print("Results saved to experiment_summary.csv and (optionally) NetCDF.")


In [14]:
import os 
#  change to specific directory for user running code 
os.chdir("/Users/sophiekim/Desktop/2_research/MamalakisResearch") 
base_path = os.getcwd()

# everyone should have locally loaded 'data' folder
data_path = base_path + '/data/'

In [12]:
model_list = [
    "CNRM_ESM2-1_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MIROC6_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MPI-ESM1-2-LR_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "MRI-ESM2-0_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
    "UKESM1-0-LL_ssp119_ssp126_201501_210012_r1-5_2pt5degree.nc",
]

In [ ]:

# only going up to 2065 for start date bc data only goes to 2100 -- but only feasible to go to 2095 
    # if went to 2075, then early period would be 2075-2084 and then late period would be 2085-2094, which kinda makes no sense (i think?)
results = run_climate_experiment(['ssp119', 'ssp126'], [2015, 2025, 2035, 2045, 2055, 2065], model_list, data_path)

Processing: ssp119 | Early: 2015 | Late: 2025
processing model: CNRM_ESM2-1


/var/folders/v1/c8q_17915_v735dr_k_2fb7w0000gn/T/ipykernel_45940/1149981826.py:92: RuntimeWarning: Mean of empty slice
  p_mean = np.nanmean(baseline_slice, axis=(0, 2))
/Users/sophiekim/miniforge3/envs/climate_env/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/var/folders/v1/c8q_17915_v735dr_k_2fb7w0000gn/T/ipykernel_45940/1149981826.py:116: RuntimeWarning: Mean of empty slice
  yearly_val = calc_func(annual_slice, axis=1)


processing model: MIROC6
processing model: MPI-ESM1-2-LR
processing model: MRI-ESM2-0
processing model: UKESM1-0-LL


2026-04-06 13:39:03.197606: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-04-06 13:39:03.197983: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-04-06 13:39:03.198369: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-04-06 13:39:03.198436: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-06 13:39:03.198912: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-04-06 13:39:04.757279: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step
Processing: ssp119 | Early: 2015 | Late: 2035
processing model: CNRM_ESM2-1


...

processing model: MIROC6
processing model: MPI-ESM1-2-LR
processing model: MRI-ESM2-0
processing model: UKESM1-0-LL
2/2 ━━━━━━━━━━━━━━━━━━━━ 17s 17s/step
Processing: ssp126 | Early: 2065 | Late: 2075
processing model: CNRM_ESM2-1


/var/folders/v1/c8q_17915_v735dr_k_2fb7w0000gn/T/ipykernel_45940/1149981826.py:92: RuntimeWarning: Mean of empty slice
  p_mean = np.nanmean(baseline_slice, axis=(0, 2))
/Users/sophiekim/miniforge3/envs/climate_env/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/var/folders/v1/c8q_17915_v735dr_k_2fb7w0000gn/T/ipykernel_45940/1149981826.py:116: RuntimeWarning: Mean of empty slice
  yearly_val = calc_func(annual_slice, axis=1)


processing model: MIROC6
processing model: MPI-ESM1-2-LR
processing model: MRI-ESM2-0
processing model: UKESM1-0-LL
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
Processing: ssp126 | Early: 2065 | Late: 2085
processing model: CNRM_ESM2-1


/var/folders/v1/c8q_17915_v735dr_k_2fb7w0000gn/T/ipykernel_45940/1149981826.py:92: RuntimeWarning: Mean of empty slice
  p_mean = np.nanmean(baseline_slice, axis=(0, 2))
/Users/sophiekim/miniforge3/envs/climate_env/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/var/folders/v1/c8q_17915_v735dr_k_2fb7w0000gn/T/ipykernel_45940/1149981826.py:116: RuntimeWarning: Mean of empty slice
  yearly_val = calc_func(annual_slice, axis=1)


processing model: MIROC6
processing model: MPI-ESM1-2-LR
processing model: MRI-ESM2-0
processing model: UKESM1-0-LL
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 472ms/step
Results saved to experiment_summary.csv and (optionally) NetCDF.
